In [ ]:
#| default_exp plots

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
from collections.abc import Callable
from typing import Any

In [ ]:
#| export
import pandas as pd

In [ ]:
#| export
import holoviews as hv
from bokeh.models import ColumnDataSource, CustomAction, CustomJS
from bokeh.models.formatters import NumeralTickFormatter

In [ ]:
#| export
import hvplot.pandas

In [ ]:
#| export
DATE_COL = "date"
SERIES_COL = "series"
VALUE_COL = "value"

In [ ]:
#| export
def _outside_legend_hook(plot: Any, element: Any) -> None:
    """Place the legend just outside the plot area on the right."""
    fig = plot.state
    legend = fig.legend
    if legend is None:
        return
    legend.location = (1.02, 1.0)
    legend.border_line_color = None
    fig.min_border_right = max(fig.min_border_right or 0, 120)

In [ ]:
#| export
def _make_hline_tool_hook(
    *,
    default_value: float,
    line_opts: dict[str, Any] | None,
) -> Callable[[Any, Any], None]:
    """Create a Bokeh toolbar action that adds horizontal reference lines."""
    opts = {"color": "gray", "line_dash": "dashed", "line_width": 1.5}
    opts.update(line_opts or {})

    def _hline_tool_hook(plot: Any, element: Any) -> None:
        fig = plot.state
        source = ColumnDataSource(data={"xs": [], "ys": []})
        renderer = fig.multi_line(
            xs="xs",
            ys="ys",
            source=source,
            line_color=opts["color"],
            line_dash=opts["line_dash"],
            line_width=opts["line_width"],
        )
        renderer.name = "interactive_hlines"
        fig.add_tools(
            CustomAction(
                description="Add horizontal line",
                callback=CustomJS(
                    args={"source": source, "plot": fig, "default_value": default_value},
                    code="""
                    const raw = prompt("Horizontal line y-value", default_value);
                    if (raw === null) {
                        return;
                    }
                    const value = raw.trim();
                    const y = value.endsWith("%")
                        ? Number(value.slice(0, -1)) / 100
                        : Number(value);
                    if (!Number.isFinite(y)) {
                        alert(`"${raw}" is not a valid number.`);
                        return;
                    }

                    const start = plot.x_range.start;
                    const end = plot.x_range.end;
                    source.data.xs = [...source.data.xs, [start, end]];
                    source.data.ys = [...source.data.ys, [y, y]];
                    source.change.emit();
                    """,
                ),
            )
        )

    return _hline_tool_hook

In [ ]:
#| export
def _make_growth_line_tool_hook(
    *,
    x_values: list[Any],
    default_rate: float,
    start_value: float,
    line_opts: dict[str, Any] | None,
) -> Callable[[Any, Any], None]:
    """Create a Bokeh toolbar action that adds compounded growth lines."""
    opts = {"color": "black", "line_dash": "dotdash", "line_width": 1.5}
    opts.update(line_opts or {})

    def _growth_line_tool_hook(plot: Any, element: Any) -> None:
        fig = plot.state
        x_source = ColumnDataSource(data={"x": x_values})
        line_source = ColumnDataSource(data={"xs": [], "ys": []})
        renderer = fig.multi_line(
            xs="xs",
            ys="ys",
            source=line_source,
            line_color=opts["color"],
            line_dash=opts["line_dash"],
            line_width=opts["line_width"],
        )
        renderer.name = "interactive_growth_lines"
        fig.add_tools(
            CustomAction(
                description="Add growth line",
                callback=CustomJS(
                    args={
                        "x_source": x_source,
                        "line_source": line_source,
                        "default_rate": default_rate,
                        "start_value": start_value,
                    },
                    code="""
                    const raw = prompt("Annual growth rate", default_rate);
                    if (raw === null) {
                        return;
                    }
                    const value = raw.trim();
                    const rate = value.endsWith("%")
                        ? Number(value.slice(0, -1)) / 100
                        : Number(value);
                    if (!Number.isFinite(rate)) {
                        alert(`"${raw}" is not a valid growth rate.`);
                        return;
                    }
                    if (rate <= -1) {
                        alert("Growth rate must be greater than -100%.");
                        return;
                    }

                    const xs = x_source.data.x;
                    if (xs.length === 0) {
                        return;
                    }

                    const toMillis = (x) => {
                        if (x instanceof Date) {
                            return x.getTime();
                        }
                        if (typeof x === "number") {
                            return x;
                        }
                        return new Date(x).getTime();
                    };

                    const yearMs = 365.25 * 24 * 60 * 60 * 1000;
                    const startMs = toMillis(xs[0]);
                    const ys = xs.map((x) => {
                        const years = (toMillis(x) - startMs) / yearMs;
                        return start_value * Math.pow(1 + rate, years);
                    });

                    line_source.data.xs = [...line_source.data.xs, xs];
                    line_source.data.ys = [...line_source.data.ys, ys];
                    line_source.change.emit();
                    """,
                ),
            )
        )

    return _growth_line_tool_hook

In [ ]:
#| export
def _clear_hline_tool_hook(plot: Any, element: Any) -> None:
    """Add a toolbar action that clears lines made by the hline tool."""
    fig = plot.state
    sources = [
        renderer.data_source
        for renderer in fig.renderers
        if getattr(renderer, "name", None) == "interactive_hlines"
    ]
    fig.add_tools(
        CustomAction(
            description="Clear horizontal lines",
            callback=CustomJS(
                args={"sources": sources},
                code="""
                for (const source of sources) {
                    source.data.xs = [];
                    source.data.ys = [];
                    source.change.emit();
                }
                """,
            ),
        )
    )

In [ ]:
#| export
def _clear_growth_line_tool_hook(plot: Any, element: Any) -> None:
    """Add a toolbar action that clears lines made by the growth line tool."""
    fig = plot.state
    sources = [
        renderer.data_source
        for renderer in fig.renderers
        if getattr(renderer, "name", None) == "interactive_growth_lines"
    ]
    fig.add_tools(
        CustomAction(
            description="Clear growth lines",
            callback=CustomJS(
                args={"sources": sources},
                code="""
                for (const source of sources) {
                    source.data.xs = [];
                    source.data.ys = [];
                    source.change.emit();
                }
                """,
            ),
        )
    )

In [ ]:
#| export
def _plot_hooks(
    *,
    legend_outside: bool,
    interactive_hlines: bool,
    hline_tool_default: float,
    hline_tool_opts: dict[str, Any] | None,
    interactive_growth_lines: bool,
    growth_line_x_values: list[Any],
    growth_line_tool_default: float,
    growth_line_start_value: float,
    growth_line_tool_opts: dict[str, Any] | None,
) -> list[Callable[[Any, Any], None]]:
    hooks: list[Callable[[Any, Any], None]] = []
    if legend_outside:
        hooks.append(_outside_legend_hook)
    if interactive_hlines:
        hooks.append(
            _make_hline_tool_hook(
                default_value=hline_tool_default,
                line_opts=hline_tool_opts,
            )
        )
        hooks.append(_clear_hline_tool_hook)
    if interactive_growth_lines:
        hooks.append(
            _make_growth_line_tool_hook(
                x_values=growth_line_x_values,
                default_rate=growth_line_tool_default,
                start_value=growth_line_start_value,
                line_opts=growth_line_tool_opts,
            )
        )
        hooks.append(_clear_growth_line_tool_hook)
    return hooks

In [ ]:
#| export
def _apply_plot_opts(
    plot: Any,
    *,
    height: int,
    max_width: int,
    responsive: bool,
    shared_axes: bool,
    hooks: list[Callable[[Any, Any], None]],
) -> Any:
    opts: dict[str, Any] = {
        "height": height,
        "responsive": responsive,
        "max_width": max_width,
        "shared_axes": shared_axes,
    }
    if hooks:
        opts["hooks"] = hooks
    return plot.opts(**opts)

In [ ]:
#| export
def _to_long_timeseries(
    data: pd.Series | pd.DataFrame,
    *,
    date_col: str = DATE_COL,
    series_col: str = SERIES_COL,
    value_col: str = VALUE_COL,
    default_series_name: str = "value",
) -> pd.DataFrame:
    """Convert a Series or wide DataFrame to a stable long-form schema."""
    if isinstance(data, pd.Series):
        series_name = str(data.name) if data.name is not None else default_series_name
        return (
            data.rename(value_col)
            .rename_axis(date_col)
            .reset_index()
            .assign(**{series_col: series_name})
            [[date_col, series_col, value_col]]
        )

    if isinstance(data, pd.DataFrame):
        return (
            data.rename_axis(index=date_col, columns=series_col)
            .reset_index()
            .melt(id_vars=date_col, var_name=series_col, value_name=value_col)
            [[date_col, series_col, value_col]]
        )

    msg = "data must be a pandas Series or DataFrame"
    raise TypeError(msg)

In [ ]:
#| export
def _value_formats(
    *,
    is_perc: bool,
    value_format: str | None,
    axis_format: str | None,
) -> tuple[str, NumeralTickFormatter | None]:
    hover_format = value_format or ("0.00%" if is_perc else "0.00")

    if axis_format is None and not is_perc:
        return hover_format, NumeralTickFormatter(format="0.00")

    formatter_format = axis_format if axis_format is not None else "0.0%"
    return hover_format, NumeralTickFormatter(format=formatter_format)

In [ ]:
#| export
def _resolve_hover_mode(
    hover: bool | str | None,
    *,
    series_count: int,
) -> bool | str:
    if hover is not None:
        return hover
    # vline attaches one tooltip per series renderer, so labels stack when lines are close
    return "mouse" if series_count > 1 else "vline"

In [ ]:
#| export
def timeseries_plot(
    data: pd.Series | pd.DataFrame,
    *,
    is_perc: bool = True,
    title: str | None = None,
    value_label: str | None = None,
    series_label: str = "Series",
    date_label: str = "Date",
    date_format: str = "%b %Y",
    value_format: str | None = None,
    axis_format: str | None = None,
    show_series_in_hover: bool | None = None,
    legend: str | bool = "right",
    legend_outside: bool = True,
    hover: bool | str | None = None,
    hline: float | None = None,
    hline_opts: dict[str, Any] | None = None,
    interactive_hlines: bool = False,
    hline_tool_default: float | None = None,
    hline_tool_opts: dict[str, Any] | None = None,
    interactive_growth_lines: bool = False,
    growth_line_tool_default: float = 0.04,
    growth_line_start_value: float = 1.0,
    growth_line_tool_opts: dict[str, Any] | None = None,
    height: int = 400,
    max_width: int = 1000,
    responsive: bool = True,
    shared_axes: bool = False,
    **kwargs: Any,
) -> Any:
    """Plot a pandas time series with consistent Bokeh hover tooltips.

    The input may be a single Series or a wide DataFrame. Both are normalized
    to ``date``, ``series``, and ``value`` columns so one tooltip definition
    works for all supported inputs.

    Hover behavior defaults to ``vline`` for a single series and ``mouse`` for
    multiple series. With ``vline``, hvPlot creates one hover tooltip per line,
    which can overlap when series are close. ``mouse`` shows only the line under
    the cursor.

    Set ``interactive_hlines=True`` to add toolbar buttons for adding and
    clearing horizontal reference lines. The add button prompts for a y-value,
    e.g. ``0.05`` or ``5%`` when plotting percentage data.

    Set ``interactive_growth_lines=True`` to add toolbar buttons for adding and
    clearing compounded growth reference lines. The add button prompts for an
    annual growth rate, e.g. ``0.04`` or ``4%``.
    """
    plot_data = _to_long_timeseries(data)
    unique_series = plot_data[SERIES_COL].nunique(dropna=False)
    include_series = show_series_in_hover
    if include_series is None:
        include_series = unique_series > 1

    hover_value_format, yformatter = _value_formats(
        is_perc=is_perc,
        value_format=value_format,
        axis_format=axis_format,
    )
    value_label = value_label or ("Return" if is_perc else "Value")
    resolved_hover = _resolve_hover_mode(hover, series_count=unique_series)

    hover_tooltips = [
        (date_label, f"@{{{DATE_COL}}}{{{date_format}}}"),
        (value_label, f"@{{{VALUE_COL}}}{{{hover_value_format}}}"),
    ]
    if include_series:
        hover_tooltips.insert(1, (series_label, f"@{{{SERIES_COL}}}"))

    growth_line_x_values = (
        plot_data[DATE_COL]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    hooks = _plot_hooks(
        legend_outside=legend_outside and legend is not False,
        interactive_hlines=interactive_hlines,
        hline_tool_default=(
            hline_tool_default
            if hline_tool_default is not None
            else 0.05
            if is_perc
            else 0
        ),
        hline_tool_opts=hline_tool_opts,
        interactive_growth_lines=interactive_growth_lines,
        growth_line_x_values=growth_line_x_values,
        growth_line_tool_default=growth_line_tool_default,
        growth_line_start_value=growth_line_start_value,
        growth_line_tool_opts=growth_line_tool_opts,
    )

    plot = plot_data.hvplot.line(
        x=DATE_COL,
        y=VALUE_COL,
        by=SERIES_COL,
        title=title,
        height=height,
        responsive=responsive,
        max_width=max_width,
        padding=0.01,
        #autorange="y",
        hover=resolved_hover,
        hover_tooltips=hover_tooltips,
        yformatter=yformatter,
        toolbar="above",
        legend=legend,
        **kwargs,
    ).opts(shared_axes=shared_axes)

    if hline is None:
        return _apply_plot_opts(
            plot,
            height=height,
            max_width=max_width,
            responsive=responsive,
            shared_axes=shared_axes,
            hooks=hooks,
        )

    line_opts = {"color": "gray", "line_dash": "dashed", "line_width": 1}
    line_opts.update(hline_opts or {})
    return _apply_plot_opts(
        plot * hv.HLine(hline).opts(**line_opts),
        height=height,
        max_width=max_width,
        responsive=responsive,
        shared_axes=shared_axes,
        hooks=hooks,
    )